# Customer Demographic Estimator

In [ ]:
from src.logic.customerAttributesEstimator import *
from IPython.display import Image, display, HTML
import os
from pathlib import Path
from typing import List

def get_all_paths(directory_path: str, recursive: bool = False) -> List[Path]:
    """
    Retrieves all file paths from a specified directory.
    
    Args:
        directory_path (str): The root folder to search.
        recursive (bool): If True, searches all subfolders. Defaults to False.
        
    Returns:
        List[Path]: A list of Path objects for every file found.
    """
    path_obj = Path(directory_path)
    
    # Check if the path actually exists to avoid errors
    if not path_obj.exists() or not path_obj.is_dir():
        return []

    if recursive:
        # rglob('*') finds all files and folders recursively
        return [p for p in path_obj.rglob('*') if p.is_file()]
    else:
        # iterdir() only looks at the immediate folder
        return [p for p in path_obj.iterdir() if p.is_file()]

def display_image_grid(image_paths, labels, cols=3):
    """
    Displays a grid of images with labels using HTML Flexbox.
    """
    html_content = '<div style="display: flex; flex-wrap: wrap; gap: 20px;">'
    
    for path, label in zip(image_paths, labels):
        # Create a container for each image + label
        item_html = f"""
        <div style="flex: 1 1 calc({100/cols}% - 20px); text-align: center; border: 1px solid #ddd; padding: 10px; border-radius: 8px;">
            <p style="font-weight: bold; margin-bottom: 8px;">{label}</p>
            <img src="{path}" style="width: 100%; height: auto; border-radius: 4px;">
        </div>
        """
        html_content += item_html
    
    html_content += '</div>'
    display(HTML(html_content))


# Usage
files = get_all_paths(os.path.join(os.getcwd(), "assets/faceDataPeople"), recursive=True)
image_files = []
image_labels = []
for file in files:
    estimation = estimateAttributes(file, 1)
    image_files.append(file)
    image_labels.append(f"SEX:{estimation[1]} AGE:{estimation[0]}")

display_image_grid(image_files, image_labels, cols=3)





# Simulate Customers

In [ ]:
from src.mockData.mock_data import *
simulate_customer(
    age="(15-20)",
    path_points=[(5, 5), (10, 15), (15, 25), (25, 30), (30, 40), (55, 10)],
    products=[chips, soda],
    sex="F",
)

# Display Customer Paths
This section uses `src/path_constructor.py` and Manim to animate every customer's path.
If Manim isn't installed, run `pip install manim` in your environment.


In [48]:
from pathlib import Path as FsPath
from collections import defaultdict
from src.database.database_setup import initialize_db, close_db, PathTable


def find_db_path(filename="store.db"):
    # Search upward from CWD for the database file.
    for candidate in [FsPath.cwd(), *FsPath.cwd().parents]:
        db_path = candidate / filename
        if db_path.exists():
            return db_path
    return None


def load_paths_from_db(db_path):
    # Load all customer paths directly from PathTable.
    initialize_db(str(db_path))
    try:
        rows = (
            PathTable
            .select(
                PathTable.customer_id,
                PathTable.timestamp,
                PathTable.location_x,
                PathTable.location_y,
                PathTable.path_id,
            )
            .order_by(
                PathTable.customer_id,
                PathTable.timestamp,
                PathTable.path_id,
            )
        )

        paths_by_customer = defaultdict(list)
        for row in rows:
            paths_by_customer[row.customer_id].append([row.location_x, row.location_y])

        return list(paths_by_customer.values())
    finally:
        close_db()


db_path = find_db_path()
if db_path is None:
    raise FileNotFoundError("store.db not found in current or parent directories")

all_paths = load_paths_from_db(db_path)
print(f"Loaded {len(all_paths)} customer paths from {db_path}")

# Keep paths in memory for inspection in the notebook.
all_paths[:3]


Loaded 6 customer paths from /Users/ryanpelchat/Documents/GitHub/project-group-12-0x5572414e657264/store.db


[[[10, 15], [15, 25], [25, 30], [30, 40], [55, 10], [5, 5]],
 [[30, 20], [45, 30], [50, 40], [55, 10], [25, 10]],
 [[15, 20], [45, 25], [50, 35], [55, 10], [10, 10]]]

In [49]:
%%manim -qm -v WARNING CustomerPathsScene
from pathlib import Path as FsPath
from collections import defaultdict
from manim import *
from src.database.database_setup import initialize_db, close_db, PathTable


def find_db_path(filename="store.db"):
    for candidate in [FsPath.cwd(), *FsPath.cwd().parents]:
        db_path = candidate / filename
        if db_path.exists():
            return db_path
    return None


class CustomerPathsScene(Scene):
    def construct(self):
        db_path = find_db_path()
        if db_path is None:
            self.add(Text("store.db not found").scale(0.7))
            self.wait(2)
            return

        initialize_db(str(db_path))
        try:
            rows = (
                PathTable
                .select(
                    PathTable.customer_id,
                    PathTable.timestamp,
                    PathTable.location_x,
                    PathTable.location_y,
                    PathTable.path_id,
                )
                .order_by(
                    PathTable.customer_id,
                    PathTable.timestamp,
                    PathTable.path_id,
                )
            )

            paths_by_customer = defaultdict(list)
            for row in rows:
                paths_by_customer[row.customer_id].append([row.location_x, row.location_y])

            paths = list(paths_by_customer.values())

            if not paths:
                self.add(Text("No customer paths found").scale(0.7))
                self.wait(2)
                return

            xs = [pt[0] for path in paths for pt in path]
            ys = [pt[1] for path in paths for pt in path]
            min_x, max_x = min(xs), max(xs)
            min_y, max_y = min(ys), max(ys)

            padding = 1
            axes = Axes(
                x_range=[min_x - padding, max_x + padding, 1],
                y_range=[min_y - padding, max_y + padding, 1],
                x_length=10,
                y_length=6,
                tips=False,
            )
            axes_labels = axes.get_axis_labels(x_label="X", y_label="Y")
            self.add(axes, axes_labels)

            colors = color_gradient(
                [BLUE, GREEN, YELLOW, ORANGE, RED, PURPLE],
                len(paths),
            )

            path_mobjects = []
            for path, color in zip(paths, colors):
                if len(path) == 1:
                    dot = Dot(axes.c2p(path[0][0], path[0][1]), color=color, radius=0.05)
                    path_mobjects.append(dot)
                    continue
                points = [axes.c2p(x, y) for x, y in path]
                vm = VMobject(color=color, stroke_width=2)
                vm.set_points_as_corners(points)
                path_mobjects.append(vm)

            self.play(
                LaggedStart(
                    *[Create(p) for p in path_mobjects],
                    lag_ratio=0.02,
                    run_time=6,
                )
            )
            self.wait(1)
        finally:
            close_db()


Manim Community v0.20.1

# Data Cleanup
Use the function below to delete all customer-related data from the database.


In [ ]:
from pathlib import Path as FsPath
from src.database.database_setup import (
    initialize_db,
    close_db,
    CustomerTable,
    PathTable,
    CheckoutTable,
    PurchaseTable,
    LogTable,
)


def delete_all_customer_data(db_path=None, include_logs=False):
    # Delete all customer-related data from the SQLite database.
    # This removes rows from: purchase, checkout, path, customer.
    # Set include_logs=True to also delete all log rows.
    if db_path is None:
        db_path = FsPath.cwd() / "store.db"

    db_path = FsPath(db_path)
    if not db_path.exists():
        raise FileNotFoundError(f"Database not found: {db_path}")

    initialize_db(str(db_path))
    try:
        deleted = {}
        deleted["purchase"] = PurchaseTable.delete().execute()
        deleted["checkout"] = CheckoutTable.delete().execute()
        deleted["path"] = PathTable.delete().execute()
        deleted["customer"] = CustomerTable.delete().execute()
        if include_logs:
            deleted["log"] = LogTable.delete().execute()
        return deleted
    finally:
        close_db()

delete_all_customer_data(include_logs=False)
